# Explore — NOAA stations & datatypes

Scratch notebook for deciding what the frozen bronze column contract should be. **Read-only:** it
calls the NOAA catalogue endpoints and pulls a sample window, but writes no tables. Safe to re-run
and safe to leave half-finished.

The question it answers is not "what datatypes exist" — NOAA advertises hundreds — but *which ones
this station actually reports densely enough over our date range to be worth a column.* Those are
very different lists, which is why section 3 exists.

Output of the last cell is the exact string to paste into the `datatypes` widget of
`10_ingest_nightly.ipynb`.

## 0. Parameters

In [ ]:
dbutils.widgets.text('location_id', 'FIPS:17031', 'NOAA locationid')
dbutils.widgets.text('station_id', '', 'Station to inspect (blank = best ranked)')
dbutils.widgets.text('start_date', '2020-01-01', 'Window start (station must span this)')
dbutils.widgets.text('end_date', '2024-12-31', 'Window end')
dbutils.widgets.text('probe_start', '2024-01-01', 'Density probe start')
dbutils.widgets.text('probe_end', '2024-12-31', 'Density probe end')
dbutils.widgets.text('min_coverage', '0.95', 'Coverage threshold to keep a datatype')
dbutils.widgets.text('max_datatypes', '25', 'Cap on datatypes probed in one request')
dbutils.widgets.text('secret_scope', 'mlo', 'Secret scope holding the NOAA token')
dbutils.widgets.text('secret_key', 'WEATHER_API_KEY', 'Secret key holding the NOAA token')

In [ ]:
import os
import sys

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import pandas as pd
from data_pipelines.noaa_client import (
    get_datatypes,
    get_ghcnd_daily,
    get_stations,
    make_headers,
    to_wide,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)

LOCATION_ID = dbutils.widgets.get('location_id')
START_DATE = dbutils.widgets.get('start_date')
END_DATE = dbutils.widgets.get('end_date')
PROBE_START = dbutils.widgets.get('probe_start')
PROBE_END = dbutils.widgets.get('probe_end')
MIN_COVERAGE = float(dbutils.widgets.get('min_coverage'))
MAX_DATATYPES = int(dbutils.widgets.get('max_datatypes'))

HEADERS = make_headers(
    dbutils.secrets.get(scope=dbutils.widgets.get('secret_scope'),
                        key=dbutils.widgets.get('secret_key'))
)

## 1. Stations in the area

Every GHCND station NOAA lists for the location, ranked by whether it spans our window and then by
`datacoverage`. Airport stations (`USW…`) are usually the dense, long-running ones; co-op and CoCoRaHS
sites (`USC…`, `US1…`) are often precipitation-only or patchy.

`mindate`/`maxdate` are the station's own reporting span — a station that closed in 2016 will still
appear here, which is why the backfill filters on the window rather than taking the first row.

In [ ]:
stations = get_stations(LOCATION_ID, headers=HEADERS)
stations['covers_window'] = (stations['mindate'] <= START_DATE) & (stations['maxdate'] >= END_DATE)

ranked = stations.sort_values(['covers_window', 'datacoverage'], ascending=False).reset_index(drop=True)

print(f'{len(stations)} GHCND stations in {LOCATION_ID}')
print(f'{int(stations["covers_window"].sum())} of them span {START_DATE}..{END_DATE}')

cols = [c for c in ['id', 'name', 'mindate', 'maxdate', 'datacoverage', 'elevation', 'covers_window']
        if c in ranked.columns]
ranked[cols].head(30)

In [ ]:
# Blank widget = take the top-ranked station. Set station_id explicitly to compare another one.
STATION_ID = dbutils.widgets.get('station_id').strip() or ranked.iloc[0]['id']
STATION_NAME = stations.loc[stations['id'] == STATION_ID, 'name'].iloc[0]

print('Inspecting:', STATION_ID, '-', STATION_NAME)

## 2. What this station reports

Scoped to the station, so it's the short honest list rather than the full GHCND catalogue.

Treat the `datacoverage` column here as a **hint, not an answer** — it describes the datatype's
coverage in the dataset broadly, not the fraction of days in *our* window that carry a value at
*this* station. Section 3 measures that directly.

In [ ]:
datatypes = get_datatypes(HEADERS, station_id=STATION_ID)

cols = [c for c in ['id', 'name', 'mindate', 'maxdate', 'datacoverage'] if c in datatypes.columns]
datatypes_ranked = datatypes.sort_values('datacoverage', ascending=False).reset_index(drop=True)

print(f'{len(datatypes)} datatypes reported by {STATION_ID}')
datatypes_ranked[cols]

## 3. Density check — the decisive one

Pulls a sample window and counts, per datatype, how many days actually came back with a value.
This is what separates "the station reports SNWD" from "SNWD is populated often enough to model on".

One year is usually enough to judge, and keeps the pull cheap — NOAA is rate-limited and the client
pages at 1000 records. Widen `probe_start`/`probe_end` if a datatype looks seasonal and you want to
see it across a full cycle.

In [ ]:
candidates = tuple(datatypes_ranked['id'].tolist()[:MAX_DATATYPES])
print(f'probing {len(candidates)} datatypes over {PROBE_START}..{PROBE_END}')

probe_raw = get_ghcnd_daily(
    station_id=STATION_ID,
    start_date=PROBE_START,
    end_date=PROBE_END,
    headers=HEADERS,
    datatypes=candidates,
)
probe = to_wide(probe_raw, datatypes=candidates)

n_days = (pd.Timestamp(PROBE_END) - pd.Timestamp(PROBE_START)).days + 1

coverage = pd.DataFrame({
    'datatype': candidates,
    'days_with_value': [int(probe[c].notna().sum()) for c in candidates],
})
coverage['coverage'] = (coverage['days_with_value'] / n_days).round(3)
coverage = (coverage
            .merge(datatypes_ranked[['id', 'name']].rename(columns={'id': 'datatype'}),
                   on='datatype', how='left')
            .sort_values('coverage', ascending=False)
            .reset_index(drop=True))

print(f'{len(probe)} station-days returned out of {n_days} calendar days')
coverage

## 4. The frozen contract

Whatever comes out of here becomes the column set the nightly `MERGE` depends on, so changing it
later means re-running the backfill. Worth being a little generous now — an extra sparse column is
cheap, a schema migration later is not.

In [ ]:
keep = coverage.loc[coverage['coverage'] >= MIN_COVERAGE, 'datatype'].tolist()
thin = coverage[(coverage['coverage'] > 0) & (coverage['coverage'] < MIN_COVERAGE)]
absent = coverage.loc[coverage['coverage'] == 0, 'datatype'].tolist()

print(f'{len(keep)} datatypes at >= {MIN_COVERAGE:.0%} coverage:')
print()
print('    ' + ','.join(keep))
print()
print('To adopt this contract:')
print('  1. paste the string above into the `datatypes` widget of 10_ingest_nightly.ipynb')
print('  2. set DEFAULT_DATATYPES in data_pipelines/noaa_client.py to match')
print('  3. re-run 01_data_ingestion.ipynb to rebuild bronze with the new columns')
print('  4. only then schedule the nightly job (its precondition check enforces this order)')

if not thin.empty:
    print()
    print('Present but sparse - include only if the downstream model tolerates nulls:')
    display(thin)

if absent:
    print()
    print(f'Reported by the station but empty over the probe window: {absent}')
    print('Could still be seasonal - widen probe_start/probe_end before writing these off.')

---

**Next:** to compare stations, set the `station_id` widget to another `id` from section 1 and re-run
from section 2. Coverage differs a lot between airport and co-op sites, and the choice is worth making
deliberately — the nightly job pins whatever station you land on.